In [1]:
!pip install statsforecast

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.6/354.6 kB 31.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.2/348.2 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.0/281.0 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.3/37.3 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.9/59.9 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3


In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_squared_log_error
import plotly.express as px
import plotly.graph_objects as go
from statsforecast import StatsForecast
from statsforecast.models import AutoETS, AutoTheta, Naive, SeasonalNaive
from catboost import CatBoostRegressor
from sklearn.metrics import mean_squared_log_error


In [4]:
holidays=pd.read_csv('/content/holidays_events.csv')
items = pd.read_csv("/content/items.csv")
oil = pd.read_csv("/content/oil.csv")
stores = pd.read_csv("/content/stores.csv")
transactions = pd.read_csv("/content/transactions.csv")

Посмотрим, что из себя представляют все таблицы

In [5]:
holidays.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 350 entries, 0 to 349
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   date         350 non-null    object
 1   type         350 non-null    object
 2   locale       350 non-null    object
 3   locale_name  350 non-null    object
 4   description  350 non-null    object
 5   transferred  350 non-null    bool  
dtypes: bool(1), object(5)
memory usage: 14.1+ KB


In [6]:
holidays.head()

,date,type,locale,locale_name,description,transferred
0,2012-03-02,Holiday,Local,Manta,Fundacion de Manta,False
1,2012-04-01,Holiday,Regional,Cotopaxi,Provincializacion de Cotopaxi,False
2,2012-04-12,Holiday,Local,Cuenca,Fundacion de Cuenca,False
3,2012-04-14,Holiday,Local,Libertad,Cantonizacion de Libertad,False
4,2012-04-21,Holiday,Local,Riobamba,Cantonizacion de Riobamba,False


In [7]:
holidays.date = pd.to_datetime(holidays.date)

In [8]:
items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4100 entries, 0 to 4099
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   item_nbr    4100 non-null   int64 
 1   family      4100 non-null   object
 2   class       4100 non-null   int64 
 3   perishable  4100 non-null   int64 
dtypes: int64(3), object(1)
memory usage: 128.3+ KB


In [9]:
items.head()

,item_nbr,family,class,perishable
0,96995,GROCERY I,1093,0
1,99197,GROCERY I,1067,0
2,103501,CLEANING,3008,0
3,103520,GROCERY I,1028,0
4,103665,BREAD/BAKERY,2712,1


In [10]:
items.item_nbr = items.item_nbr.astype('int32')
items['class'] = items['class'].astype('int8')
items.perishable = items.perishable.astype('int8')

In [11]:
oil.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1218 entries, 0 to 1217
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   date        1218 non-null   object 
 1   dcoilwtico  1175 non-null   float64
dtypes: float64(1), object(1)
memory usage: 19.2+ KB


In [12]:
oil.head()

,date,dcoilwtico
0,2013-01-01,NaN
1,2013-01-02,93.14
2,2013-01-03,92.97
3,2013-01-04,93.12
4,2013-01-07,93.20


In [13]:
oil.date = pd.to_datetime(oil.date)
oil.dcoilwtico = oil.dcoilwtico.astype('float32')

In [14]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   store_nbr  54 non-null     int64 
 1   city       54 non-null     object
 2   state      54 non-null     object
 3   type       54 non-null     object
 4   cluster    54 non-null     int64 
dtypes: int64(2), object(3)
memory usage: 2.2+ KB


In [15]:
stores.head()

,store_nbr,city,state,type,cluster
0,1,Quito,Pichincha,D,13
1,2,Quito,Pichincha,D,13
2,3,Quito,Pichincha,D,8
3,4,Quito,Pichincha,D,9
4,5,Santo Domingo,Santo Domingo de los Tsachilas,D,4


In [16]:
stores.store_nbr = stores.store_nbr.astype('int8')
stores.cluster = stores.cluster.astype('int8')

In [17]:
transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 83488 entries, 0 to 83487
Data columns (total 3 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   date          83488 non-null  object
 1   store_nbr     83488 non-null  int64 
 2   transactions  83488 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.9+ MB


In [18]:
transactions.head()

,date,store_nbr,transactions
0,2013-01-01,25,770
1,2013-01-02,1,2111
2,2013-01-02,2,2358
3,2013-01-02,3,3487
4,2013-01-02,4,1922


In [19]:
transactions.date = pd.to_datetime(transactions.date)
transactions.store_nbr = transactions.store_nbr.astype('int8')
transactions.transactions = transactions.transactions.astype('int16')

Train.csv большой, поэтому посмотрим какие есть колонки, каких типов.

In [20]:
!7z x "/content/train.csv.7z" -o"/content"


7-Zip [64] 16.02 : Copyright (c) 1999-2016 Igor Pavlov : 2016-05-21
p7zip Version 16.02 (locale=en_US.UTF-8,Utf16=on,HugeFiles=on,64 bits,2 CPUs Intel(R) Xeon(R) CPU @ 2.00GHz (50653),ASM,AES-NI)

Scanning the drive for archives:
  0M Scan /content/                   1 file, 474092593 bytes (453 MiB)

Extracting archive: /content/train.csv.7z
--
Path = /content/train.csv.7z
Type = 7z
Physical Size = 474092593
Headers Size = 122
Method = LZMA2:24
Solid = -
Blocks = 1

  0%      0% - train.csv                  1% - train.csv                  2% - train.csv                  3% - train.csv                  4% - train.csv                  5% - train.csv                  6% - train.csv                  7% - train.csv

In [21]:
df_sample = pd.read_csv('train.csv', nrows=1000)
df_sample.head()

,id,date,store_nbr,item_nbr,unit_sales,onpromotion
0,0,2013-01-01,25,103665,7.0,NaN
1,1,2013-01-01,25,105574,1.0,NaN
2,2,2013-01-01,25,105575,2.0,NaN
3,3,2013-01-01,25,108079,1.0,NaN
4,4,2013-01-01,25,108701,1.0,NaN


In [22]:
df_sample.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           1000 non-null   int64  
 1   date         1000 non-null   object 
 2   store_nbr    1000 non-null   int64  
 3   item_nbr     1000 non-null   int64  
 4   unit_sales   1000 non-null   float64
 5   onpromotion  0 non-null      float64
dtypes: float64(2), int64(3), object(1)
memory usage: 47.0+ KB


Проверим пропуски

In [23]:
print("Пропуски в stores:")
print(stores.isnull().sum())
print("\nПропуски в items:")
print(items.isnull().sum())
print("\nПропуски в transactions:")
print(transactions.isnull().sum())
print('\nПропуски в oil:\n', oil.isnull().sum())
print('\nПропуски в holidays:\n', holidays.isnull().sum())

Пропуски в stores:
store_nbr    0
city         0
state        0
type         0
cluster      0
dtype: int64

Пропуски в items:
item_nbr      0
family        0
class         0
perishable    0
dtype: int64

Пропуски в transactions:
date            0
store_nbr       0
transactions    0
dtype: int64

Пропуски в oil:
 date           0
dcoilwtico    43
dtype: int64

Пропуски в holidays:
 date           0
type           0
locale         0
locale_name    0
description    0
transferred    0
dtype: int64


Цена нефти – важный признак. Заполним пропуски линейной интерполяцией.

In [24]:
oil = oil.set_index('date').sort_index()
oil['dcoilwtico'] = oil['dcoilwtico'].interpolate(method='linear')
oil = oil.reset_index()
print('Пропусков в oil:', oil['dcoilwtico'].isnull().sum())

Пропусков в oil: 1


Остался один пропуск самый первый, выкинем его

In [25]:
oil.dropna(inplace=True)

Метрика RMSLE была выбрана, так как именно она указана в соревновании на Kaggle.

Главное отличие от обычного RMSE — мы считаем ошибку в логарифмической шкале.
В реальных данных большинство товаров продаются в небольших количествах, а некоторые редко и помногу. Если мы будем использовать обычный RMSE, огромные ошибки на редких, но крупных продажах перекроют ошибки на множестве мелких. RMSLE уравнивает вклад маленьких и больших значений в общую ошибку, то есть оценивает точность прогноза относительно масштаба продаж.

Кроме того, RMSLE штрафует занижение сильнее, чем завышение потому что логарифм меняется быстрее при малых значениях. С точки зрения бизнеса это хорошо: недооценить спрос опаснее, чем переоценить, т.к. дефицит товара ведёт к потерянным продажам и недовольству клиентов, тогда как излишек — просто дополнительные запасы.

In [26]:
def rmsle(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred))**2))

Так как датасет огромный, примерно 125 млн. строк, все считать мы не можем. Поэтому будем рассматривать только данные с января 2017 года. Одной порцией тоже считать не получается, поэтому будем считывать кусочками и сразу приводить у минимально возможным типам.

In [27]:
start_date = '2017-01-01'
chunk_size = 1000000

chunks = []
for chunk in pd.read_csv('train.csv',
                         index_col=0,
                         parse_dates=['date'],
                         chunksize=chunk_size):
    chunk.store_nbr = chunk.store_nbr.astype('int8')
    chunk.item_nbr = chunk.item_nbr.astype('int32')
    chunk.unit_sales = chunk.unit_sales.astype('float32')
    chunk['onpromotion'] =chunk['onpromotion'].fillna(0)
    chunk.onpromotion = chunk.onpromotion.astype('int8')
    chunk = chunk[chunk['date'] >= start_date]
    if not chunk.empty:
        chunks.append(chunk)

df = pd.concat(chunks, ignore_index=True)
df['unit_sales'] = df['unit_sales'].clip(lower=0)


/tmp/ipykernel_6473/1349361218.py:5: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  for chunk in pd.read_csv('train.csv',


И бейзлайн и ML модели могут отработать на большом количестве рядов. У меня получалось около 160тыс. Но для трансформера это оказалось очень много, столько времени и ресурсов для расчета нет. Поэтому в данном случае я решила ограничиться 20 тыс. рядов.

In [28]:
row_num = 20000
series_length = df.groupby(['store_nbr', 'item_nbr']).size().reset_index(name='count')
series_length = series_length.sort_values('count', ascending=False)
top_series = series_length.head(row_num)[['store_nbr', 'item_nbr']]

df = df.merge(top_series, on=['store_nbr', 'item_nbr'], how='inner')

In [31]:
from tqdm import tqdm
min_date = df['date'].min()
max_date = df['date'].max()
all_dates = pd.date_range(start=min_date, end=max_date, freq='D')

filled_groups = []

for (store, item), group in tqdm(df.groupby(['store_nbr', 'item_nbr']), desc="Обработка групп"):
    group = group.set_index('date')
    group = group.reindex(all_dates)
    group['store_nbr'] = store
    group['item_nbr'] = item
    group['unit_sales'] = group['unit_sales'].fillna(0)
    group['onpromotion'] = group['onpromotion'].fillna(0)
    group = group.reset_index().rename(columns={'index': 'date'})
    filled_groups.append(group)

df = pd.concat(filled_groups, ignore_index=True)

Обработка групп: 100%|██████████| 20000/20000 [00:39<00:00, 505.49it/s]


In [32]:
split_date = df['date'].max() - pd.Timedelta(days=16)
train_df = df[df['date'] <= split_date].copy()
val_df = df[df['date'] > split_date].copy()

print(f"Train: {len(train_df)} записей")
print(f"Validation: {len(val_df)} записей")

Train: 4220000 записей
Validation: 320000 записей


Посчитаем бейзлайн.

Средние продажи по каждой паре (store, item)

In [33]:
item_store_mean = train_df.groupby(['store_nbr', 'item_nbr'], as_index=False)['unit_sales'].mean()
item_store_mean.rename(columns={'unit_sales': 'mean_sales'}, inplace=True)
val_df = val_df.merge(item_store_mean, on=['store_nbr', 'item_nbr'], how='left')
val_df['mean_sales'] = val_df['mean_sales'].fillna(0)

Наивный сезонный прогноз в нашем исполнении

In [34]:
T = train_df['date'].max()
last_week = train_df[train_df['date'] > T - pd.Timedelta(days=7)].copy()
last_week['dayofweek'] = last_week['date'].dt.dayofweek
last_week.rename(columns={'unit_sales': 'naive_week_repeat'}, inplace=True)
val_df['dayofweek'] = val_df['date'].dt.dayofweek
val_df = val_df.merge(last_week[['store_nbr', 'item_nbr', 'dayofweek', 'naive_week_repeat']],
                      on=['store_nbr', 'item_nbr', 'dayofweek'],how='left')

val_df['naive_week_repeat'] = val_df['naive_week_repeat'].fillna(0)

In [35]:
y_true = val_df['unit_sales'].values
rmsle_naive = rmsle(y_true, val_df['naive_week_repeat'].values)
rmsle_mean = rmsle(y_true, val_df['mean_sales'].values)

print("Прогноз средним: RMSLE =", rmsle_mean)
print("Наивный прогноз: RMSLE =", rmsle_naive)

Прогноз средним: RMSLE = 0.6099922
Наивный прогноз: RMSLE = 0.7252088


Использование statsforecast

In [36]:
train_df['unique_id'] = train_df['store_nbr'].astype(str) + '_' + train_df['item_nbr'].astype(str)
val_df['unique_id'] = val_df['store_nbr'].astype(str) + '_' + val_df['item_nbr'].astype(str)

train_sf = train_df[['unique_id', 'date', 'unit_sales']].rename(columns={'date': 'ds', 'unit_sales': 'y'})
test_sf = val_df[['unique_id', 'date', 'unit_sales']].rename(columns={'date': 'ds', 'unit_sales': 'y'})

Выбираем модели. Учитывая достаточно большое количество рядов обучить получилось только AutoTheta, Naive и SeasonalNaive

In [42]:
models = [#AutoETS(season_length=7),
          AutoTheta(season_length=7),
          Naive(),
          SeasonalNaive(season_length=7)]

In [44]:
sf = StatsForecast(models=models, freq='D', n_jobs=-1, verbose=True)

sf.fit(df=train_sf)

StatsForecast(models=[AutoTheta,Naive,SeasonalNaive])

In [45]:
HORIZON = 16
forecasts = sf.predict(h=HORIZON)

In [48]:
forecasts_long = forecasts.melt(id_vars=['unique_id', 'ds'],var_name='model',value_name='forecast')
eval_df = test_sf.merge(forecasts_long, on=['unique_id', 'ds'], how='inner')

results = []
for model in [m.__class__.__name__ for m in models]:
    model_preds = eval_df[eval_df['model'] == model]
    if len(model_preds) > 0:
        score = rmsle(model_preds['y'].values, np.maximum(model_preds['forecast'].values,0))
        results.append({'model': model, 'rmsle': score})


In [71]:
for i in range(len(results)):
  print(results[i]['model'], f"{results[i]['rmsle']:.4f}")


AutoTheta 0.6237
Naive 0.7809
SeasonalNaive 0.7252


In [ ]:
del train_sf,test_sf
gc.collect()

In [ ]:
del train_df,val_df
gc.collect()

Перейдем к ML модели, будем использовать Catboost.

In [53]:
val_end = df['date'].max()
val_start = val_end - pd.DateOffset(days=15)

df['year'] = df['date'].dt.year.astype('int16')
df['month'] = df['date'].dt.month.astype('int8')
df['day'] = df['date'].dt.day.astype('int8')
df['dayofweek'] = df['date'].dt.dayofweek.astype('int8')
df['weekend'] = (df['dayofweek'] >= 5).astype('boolean')

for lag in [1, 7, 14, 28]:
        df[f'lag_{lag}'] = df.groupby(['store_nbr', 'item_nbr'])['unit_sales'].shift(lag)

train_feat = df[df['date'] < val_start].copy()
val_feat = df[df['date'] >= val_start].copy()

feature_cols = ['store_nbr', 'item_nbr','year', 'month', 'day', 'dayofweek', 'weekend',
                'lag_1', 'lag_7', 'lag_14', 'lag_28']

for col in ['lag_1', 'lag_7', 'lag_14', 'lag_28']:
    train_feat[col] = train_feat[col].fillna(0)
    val_feat[col] = val_feat[col].fillna(0)

In [54]:
y_train = np.log1p(train_feat['unit_sales'].values)
y_val = np.log1p(val_feat['unit_sales'].values)
X_train = train_feat[feature_cols]
X_val = val_feat[feature_cols]

In [55]:
import gc
del df
gc.collect()

574

In [56]:
cat_features = ['store_nbr', 'item_nbr','year', 'month', 'day', 'dayofweek', 'weekend']

Обучим Catboost без экзогенных признаков

In [57]:
model = CatBoostRegressor(
    iterations=125,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42,
    task_type='GPU'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 0.8882481	test: 0.8547206	best: 0.8547206 (0)	total: 153ms	remaining: 19s
100:	learn: 0.5331768	test: 0.5262442	best: 0.5262442 (100)	total: 7.07s	remaining: 1.68s
124:	learn: 0.5306648	test: 0.5247435	best: 0.5247435 (124)	total: 8.6s	remaining: 0us
bestTest = 0.5247434734
bestIteration = 124


CatBoostRegressor(cat_features=['store_nbr', 'item_nbr', 'year', 'month', 'day', 'dayofweek', 'weekend'], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=125, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [72]:
pred_log = model.predict(X_val)
pred_sales = np.expm1(pred_log)
true_sales = val_feat['unit_sales'].values
catboost_rmsle = rmsle(true_sales, pred_sales)
print("CatBoost RMSLE без экзогенных признаков: ", f"{catboost_rmsle:.4f}")

CatBoost RMSLE без экзогенных признаков:  0.5247


In [73]:
del y_train,y_val,X_train,X_val
gc.collect()

561

Добавим признаков из датасетов items и stores.

In [74]:

train_feat = train_feat.merge(items[['item_nbr', 'family', 'perishable', 'class']], on='item_nbr', how='left')
val_feat = val_feat.merge(items[['item_nbr', 'family', 'perishable', 'class']], on='item_nbr', how='left')

train_feat = train_feat.merge(stores, on='store_nbr', how='left')
val_feat = val_feat.merge(stores, on='store_nbr', how='left')

static_features = ['family', 'perishable', 'class', 'city', 'state', 'type', 'cluster']
feature_cols.extend(static_features)

cat_features.extend(['family', 'perishable', 'city', 'state', 'type', 'cluster'])

for col in static_features:
    if col in cat_features:
        train_feat[col] = train_feat[col].fillna('unknown')
        val_feat[col] = val_feat[col].fillna('unknown')
    else:
        train_feat[col] = train_feat[col].fillna(0)
        val_feat[col] = val_feat[col].fillna(0)

In [75]:
y_train = np.log1p(train_feat['unit_sales'].values)
y_val = np.log1p(val_feat['unit_sales'].values)
X_train = train_feat[feature_cols]
X_val = val_feat[feature_cols]

In [76]:
model = CatBoostRegressor(
    iterations=125,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42,
    task_type='GPU'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 0.8882481	test: 0.8547206	best: 0.8547206 (0)	total: 78.8ms	remaining: 9.77s
100:	learn: 0.5333628	test: 0.5260327	best: 0.5260327 (100)	total: 8.35s	remaining: 1.98s
124:	learn: 0.5307500	test: 0.5245336	best: 0.5245336 (124)	total: 10.5s	remaining: 0us
bestTest = 0.5245335772
bestIteration = 124


CatBoostRegressor(cat_features=['store_nbr', 'item_nbr', 'year', 'month', 'day', 'dayofweek', 'weekend', 'family', 'perishable', 'city', 'state', 'type', 'cluster'], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=125, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [78]:
pred_log = model.predict(X_val)
pred_sales = np.expm1(pred_log)
true_sales = val_feat['unit_sales'].values
catboost_rmsle = rmsle(true_sales, pred_sales)
print("CatBoost RMSLE с данными из датасетов items и stores:", f"{catboost_rmsle:.4f}")

CatBoost RMSLE с данными из датасетов items и stores: 0.5245


Добавим к признакая национальные праздники

In [79]:
national_holidays = holidays[holidays['locale'] == 'National']['date'].unique()
train_feat['is_national_holiday'] = train_feat['date'].isin(national_holidays).astype('boolean')
val_feat['is_national_holiday'] = val_feat['date'].isin(national_holidays).astype('boolean')

feature_cols.append('is_national_holiday')
cat_features.append('is_national_holiday')

In [80]:
del y_train,y_val,X_train,X_val
gc.collect()

45

In [81]:
y_train = np.log1p(train_feat['unit_sales'].values)
y_val = np.log1p(val_feat['unit_sales'].values)
X_train = train_feat[feature_cols]
X_val = val_feat[feature_cols]

In [82]:
model = CatBoostRegressor(
    iterations=125,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42,
    task_type='GPU'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 0.8882481	test: 0.8547206	best: 0.8547206 (0)	total: 70.8ms	remaining: 8.77s
100:	learn: 0.5323312	test: 0.5270682	best: 0.5270682 (100)	total: 6.27s	remaining: 1.49s
124:	learn: 0.5293041	test: 0.5251602	best: 0.5251602 (124)	total: 7.84s	remaining: 0us
bestTest = 0.5251602248
bestIteration = 124


CatBoostRegressor(cat_features=['store_nbr', 'item_nbr', 'year', 'month', 'day', 'dayofweek', 'weekend', 'family', 'perishable', 'city', 'state', 'type', 'cluster', 'is_national_holiday'], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=125, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [83]:
pred_log = model.predict(X_val)
pred_sales = np.expm1(pred_log)
true_sales = val_feat['unit_sales'].values
catboost_rmsle = rmsle(true_sales, pred_sales)
print(f"CatBoost RMSLE c праздниками: {catboost_rmsle:.4f}")

CatBoost RMSLE c праздниками: 0.5252


Праздники не добавили качества уберем их.

In [104]:
train_feat.drop('is_national_holiday',axis=1, inplace=True)
val_feat.drop('is_national_holiday',axis=1, inplace=True)
del feature_cols[-1]

К экзогенным признакам добавим нефть.

In [84]:
mask = (oil['date'] >= '2016-01-01') & (oil['date'] <= '2016-12-31')
oil_period = oil.loc[mask]
oil_mean_price = oil_period['dcoilwtico'].mean()

train_feat = train_feat.merge(oil, on='date', how='left')
val_feat = val_feat.merge(oil, on='date', how='left')

train_feat['dcoilwtico'] = train_feat['dcoilwtico'].fillna(oil_mean_price)
val_feat['dcoilwtico'] = val_feat['dcoilwtico'].fillna(oil_mean_price)

feature_cols.append('dcoilwtico')

In [116]:
del y_train,y_val,X_train,X_val
gc.collect()

0

In [117]:
y_train = np.log1p(train_feat['unit_sales'].values)
y_val = np.log1p(val_feat['unit_sales'].values)
X_train = train_feat[feature_cols]
X_val = val_feat[feature_cols]

In [118]:
model = CatBoostRegressor(
    iterations=125,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42,
    task_type='GPU'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 0.8882481	test: 0.8547206	best: 0.8547206 (0)	total: 73.1ms	remaining: 9.06s
100:	learn: 0.5332110	test: 0.5263448	best: 0.5263448 (100)	total: 6.78s	remaining: 1.61s
124:	learn: 0.5305752	test: 0.5249468	best: 0.5249468 (124)	total: 8.94s	remaining: 0us
bestTest = 0.5249467746
bestIteration = 124


CatBoostRegressor(cat_features=['store_nbr', 'item_nbr', 'year', 'month', 'day', 'dayofweek', 'weekend', 'family', 'perishable', 'city', 'state', 'type', 'cluster'], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=125, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [119]:
pred_log = model.predict(X_val)
pred_sales = np.expm1(pred_log)
true_sales = val_feat['unit_sales'].values
catboost_rmsle = rmsle(true_sales, pred_sales)
print(f"CatBoost RMSLE с нефтью: {catboost_rmsle:.4f}")

CatBoost RMSLE с нефтью: 0.5249


Добавим информацию о покупках

In [89]:
transactions = transactions.sort_values(['store_nbr', 'date'])
transactions['transactions_lag_1'] = transactions.groupby('store_nbr')['transactions'].shift(1)
train_feat = train_feat.merge(transactions[['date', 'store_nbr', 'transactions_lag_1']],
                              on=['date', 'store_nbr'], how='left')
val_feat = val_feat.merge(transactions[['date', 'store_nbr', 'transactions_lag_1']],
                          on=['date', 'store_nbr'], how='left')

feature_cols.append('transactions_lag_1')
train_feat['transactions_lag_1'].fillna(0, inplace=True)
val_feat['transactions_lag_1'].fillna(0, inplace=True)

/tmp/ipykernel_6473/3593836027.py:9: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_feat['transactions_lag_1'].fillna(0, inplace=True)
/tmp/ipykernel_6473/3593836027.py:10: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=Tru

In [90]:
del y_train,y_val,X_train,X_val
gc.collect()

0

In [91]:
y_train = np.log1p(train_feat['unit_sales'].values)
y_val = np.log1p(val_feat['unit_sales'].values)
X_train = train_feat[feature_cols]
X_val = val_feat[feature_cols]

In [92]:
model = CatBoostRegressor(
    iterations=125,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42,
    task_type='GPU'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 0.8882481	test: 0.8547206	best: 0.8547206 (0)	total: 71.1ms	remaining: 8.82s
100:	learn: 0.5318508	test: 0.5278506	best: 0.5278506 (100)	total: 5.24s	remaining: 1.25s
124:	learn: 0.5291725	test: 0.5262881	best: 0.5262881 (124)	total: 6.44s	remaining: 0us
bestTest = 0.5262881101
bestIteration = 124


CatBoostRegressor(cat_features=['store_nbr', 'item_nbr', 'year', 'month', 'day', 'dayofweek', 'weekend', 'family', 'perishable', 'city', 'state', 'type', 'cluster', 'is_national_holiday'], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=125, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [93]:
pred_log = model.predict(X_val)
pred_sales = np.expm1(pred_log)
true_sales = val_feat['unit_sales'].values
catboost_rmsle = rmsle(true_sales, pred_sales)
print(f"CatBoost RMSLE с информацией о покупках: {catboost_rmsle:.4f}")

CatBoost RMSLE с информацией о покупках: 0.5263


Транзакции сделали хуже, выкинем их.

In [94]:
train_feat.drop('transactions_lag_1',axis=1, inplace=True)
val_feat.drop('transactions_lag_1',axis=1, inplace=True)
feature_cols=feature_cols[:-1]

In [95]:
del y_train,y_val,X_train,X_val,transactions
gc.collect()

0

Создадим еще одну переменную - количество дней до праздника. Обычно люди покупают что либо перед праздниками.

In [120]:
all_dates_df = pd.DataFrame({'date': pd.date_range(start=train_feat['date'].min(),
                            end=val_feat['date'].max(), freq='D')})

holidays_df = pd.DataFrame({'holiday_date': national_holidays})
holidays_df = holidays_df.sort_values('holiday_date')

all_dates_df = pd.merge_asof(all_dates_df,holidays_df,left_on='date',
                             right_on='holiday_date',direction='forward')

all_dates_df['days_to_holiday'] = (all_dates_df['holiday_date'] - all_dates_df['date']).dt.days.astype('int16')
all_dates_df['days_to_holiday']=all_dates_df['days_to_holiday'].fillna(365)

train_feat = train_feat.merge(all_dates_df[['date', 'days_to_holiday']], on='date', how='left')
val_feat = val_feat.merge(all_dates_df[['date', 'days_to_holiday']], on='date', how='left')

feature_cols.append('days_to_holiday')
cat_features.append('days_to_holiday')

In [121]:
y_train = np.log1p(train_feat['unit_sales'].values)
y_val = np.log1p(val_feat['unit_sales'].values)
X_train = train_feat[feature_cols]
X_val = val_feat[feature_cols]

In [122]:
model = CatBoostRegressor(
    iterations=125,
    learning_rate=0.05,
    depth=6,
    loss_function='RMSE',
    eval_metric='RMSE',
    cat_features=cat_features,
    early_stopping_rounds=50,
    verbose=100,
    random_seed=42,
    task_type='GPU'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), use_best_model=True)

0:	learn: 0.8882481	test: 0.8547206	best: 0.8547206 (0)	total: 71.9ms	remaining: 8.92s
100:	learn: 0.5319111	test: 0.5262489	best: 0.5262489 (100)	total: 6.88s	remaining: 1.63s
124:	learn: 0.5290167	test: 0.5245741	best: 0.5245741 (124)	total: 9.11s	remaining: 0us
bestTest = 0.5245740925
bestIteration = 124


CatBoostRegressor(cat_features=['store_nbr', 'item_nbr', 'year', 'month', 'day', 'dayofweek', 'weekend', 'family', 'perishable', 'city', 'state', 'type', 'cluster', 'days_to_holiday'], depth=6, early_stopping_rounds=50, eval_metric='RMSE', iterations=125, learning_rate=0.05, loss_function='RMSE', random_seed=42, task_type='GPU', verbose=100)

In [123]:
pred_log = model.predict(X_val)
pred_sales = np.expm1(pred_log)
true_sales = val_feat['unit_sales'].values
catboost_rmsle = rmsle(true_sales, pred_sales)
print(f"CatBoost RMSLE с учетом количества дней до праздника: {catboost_rmsle:.4f}")

CatBoost RMSLE с учетом количества дней до праздника: 0.5246


Метрика немного уменьшилась, но не превзошла датасет с добавление магазинов и товаров